Hybrid Retriever - Dense and Sparse Retriever Combined

In [2]:
# libraries
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_core.documents import Document

C:\Users\Dell\AppData\Local\Temp\ipykernel_8672\3506989670.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [3]:
# sample documents
docs = [
    Document(page_content = "LangChain helps build LLM applications"),
    Document(page_content = "Pinecone is a vector database for semantic search"),
    Document(page_content = "The Effiel tower is located in Paris"),
    Document(page_content = "Langchain can be used to develop agentic AI applications"),
    Document(page_content= "Langchain has many type of retrievers")
]

Building the Dense Retriever

In [4]:
embedding_model=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
dense_vectorstore = FAISS.from_documents(docs,embedding_model)
dense_retriever = dense_vectorstore.as_retriever()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Sparse Retriever

In [6]:
sparse_retriever = BM25Retriever.from_documents(docs)
sparse_retriever.k = 3 

Combining both with Ensemble Retriever

In [8]:
hybrid_retriever = EnsembleRetriever(
    retrievers=[dense_retriever,sparse_retriever],
    weight = [0.7,0.3]
)
hybrid_retriever

EnsembleRetriever(retrievers=[VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001F6BBF552B0>, search_kwargs={}), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x000001F6BBF56510>, k=3)], weights=[0.5, 0.5])

testing

In [9]:
query = "How can i build an application using LLMs?"
results = hybrid_retriever.invoke(query)

for i,doc in enumerate(results):
    print(f"\nDocument{i+1}: \n{doc.page_content}")


Document1: 
LangChain helps build LLM applications

Document2: 
Langchain can be used to develop agentic AI applications

Document3: 
Langchain has many type of retrievers

Document4: 
Pinecone is a vector database for semantic search


RAG Pipeline w Hybrid Retriever

In [10]:
#libraries
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain

prompt template setup

In [11]:
prompt = PromptTemplate.from_template("""
Answer the question based on the context below.
Context:
{context}

Question: {input}""")

initializing LLM

In [16]:
llm = init_chat_model("groq:llama-3.1-8b-instant")
llm

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001F6BE6C2A50>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001F6BE33C1A0>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

creating stuff document chain

In [17]:
document_chain = create_stuff_documents_chain(llm=llm,prompt=prompt)
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\nAnswer the question based on the context below.\nContext:\n{context}\n\nQuestion: {input}')
| ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001F6BE6C2A50>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001F6BE33C1A0>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)
| StrOutputParser(), kwargs={},

final rag chain

In [18]:
rag_chain = create_retrieval_chain(retriever=hybrid_retriever,combine_docs_chain=document_chain)
rag_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | EnsembleRetriever(retrievers=[VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001F6BBF552B0>, search_kwargs={}), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x000001F6BBF56510>, k=3)], weights=[0.5, 0.5]), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\nAnswer the question based on the context below.\nContext:\n{context}\n\nQuestion: {input}')
            | ChatGroq(output_versio

testing out RAG Chain

In [20]:
query = {"input":"How can I build an app using LLMs?"}
response = rag_chain.invoke(query)
print("Answer:\n",response["answer"])
print("\nSource Documents:")
for i,doc in enumerate(response["context"]):
    print(f"\nDoc {i+1}: {doc.page_content}")

Answer:
 You can build an app using LLMs (Large Language Models) with the help of LangChain, which provides a platform for developing agentic AI applications. To do this, you can follow these general steps:

1. **Choose a use case**: Identify the type of application you want to build, such as a chatbot, text summarizer, or language translator.
2. **Select a retriever**: LangChain offers various types of retrievers, which are used to retrieve relevant information from a database or knowledge graph. You can choose from options like Pinecone (a vector database for semantic search) or other database integrations.
3. **Integrate with a LLM**: Use LangChain to integrate with a pre-trained LLM, such as a model from the Hugging Face Transformers library.
4. **Develop the application logic**: Write code to interact with the retriever and LLM, and define the application's logic and behavior.
5. **Deploy the application**: Use LangChain to deploy your application, which can be done through variou